### # **1. Understand the Business Model**
- Customers → Orders → Order Items ← Products
-     1 Customer can place many orders:-(1 : N)
-     1 Order contains many items:-(1 : N)
-     Many order items can refer to one product:-(N:1)
# Rules
-     Insert parent tables first:
      customers, products → orders → order_items
-     Delete child tables first:
      order_items → orders → customers/products
-     order_items is a bridge table handling the many-to-many relationship between orders and products.

# Indexes

1. customers(city, state)
   → Faster customer filtering by location.

2. products(category)
   → Faster category-wise product searches.

3. orders(order_date, status)
   → Faster date-based and status-based queries.

### 1>Creating and Inserting the Tables

In [0]:
CREATE TABLE customers (
    customer_id   INT           PRIMARY KEY,
    first_name    VARCHAR(50)   NOT NULL,
    last_name     VARCHAR(50)   NOT NULL,
    email         VARCHAR(100)  UNIQUE NOT NULL,
    city          VARCHAR(50)   NOT NULL,
    state         VARCHAR(50)   NOT NULL,
    join_date     DATE          NOT NULL,
    is_premium    BOOLEAN       NOT NULL
)
-- This adds the default constraint right after creating the table setup
ALTER TABLE customers ALTER COLUMN is_premium SET DEFAULT false;

In [0]:
%sql
-- =====================================
-- CREATE TABLE: products
-- =====================================
CREATE TABLE products ( 
    product_id    INT           PRIMARY KEY, 
    product_name  VARCHAR(100)  NOT NULL, 
    category      VARCHAR(50)   NOT NULL, 
    brand         VARCHAR(50)   NOT NULL, 
    unit_price    DECIMAL(10,2) NOT NULL  CHECK (unit_price > 0), 
    stock_qty     INT           NOT NULL  CHECK (stock_qty >= 0) 
)
ALTER TABLE products ALTER COLUMN stock_qty SET DEFAULT 0;


-- =====================================
-- CREATE TABLE: orders
-- =====================================
CREATE TABLE orders ( 
    order_id      INT           PRIMARY KEY, 
    customer_id   INT           NOT NULL, 
    order_date    DATE          NOT NULL, 
    status        VARCHAR(20)   NOT NULL CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')), 
    total_amount  DECIMAL(12,2) NOT NULL  CHECK (total_amount >= 0), 
     
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id) 
)
ALTER TABLE orders ALTER COLUMN status SET DEFAULT 'Pending';


-- =====================================
-- CREATE TABLE: order_items
-- =====================================
CREATE TABLE order_items ( 
    item_id       INT           PRIMARY KEY, 
    order_id      INT           NOT NULL, 
    product_id    INT           NOT NULL, 
    quantity      INT           NOT NULL  CHECK (quantity > 0), 
    unit_price    DECIMAL(10,2) NOT NULL  CHECK (unit_price > 0), 
    discount_pct  DECIMAL(5,2)  CHECK (discount_pct BETWEEN 0 AND 100), 
     
    FOREIGN KEY (order_id)   REFERENCES orders(order_id), 
    FOREIGN KEY (product_id) REFERENCES products(product_id) 
)

ALTER TABLE order_items ALTER COLUMN discount_pct SET DEFAULT 0;

CREATE INDEX idx_customers_city ON customers(city); 
CREATE INDEX idx_customers_state ON customers(state);

-- Index for filtering by category 
CREATE INDEX idx_products_category ON products(category);

-- Index for filtering by city/state 
CREATE INDEX idx_customers_city ON customers(city); 
CREATE INDEX idx_customers_state ON customers(state);

Adding Indexes:-

In [0]:
-- ========== INSERT: customers ========== 
INSERT INTO customers VALUES 
(101, 'Aarav',  'Sharma', 'aarav.s@email.com',  'Mumbai',    'Maharashtra', '2024-01-15', TRUE), 
(102, 'Priya',  'Patel',  'priya.p@email.com',  'Ahmedabad', 'Gujarat',     '2024-02-20', FALSE), 
(103, 'Rohan',  'Gupta',  'rohan.g@email.com',  'Delhi',     'Delhi',       '2024-03-10', TRUE), 
(104, 'Sneha',  'Reddy',  'sneha.r@email.com',  'Hyderabad', 'Telangana',   '2024-04-05', FALSE), 
(105, 'Vikram', 'Singh',  'vikram.s@email.com', 'Jaipur',    'Rajasthan',   '2024-05-12', TRUE), 
(106, 'Ananya', 'Iyer',   'ananya.i@email.com', 'Chennai',   'Tamil Nadu',  '2024-06-18', FALSE), 
(107, 'Karan',  'Mehta',  'karan.m@email.com',  'Pune',      'Maharashtra', '2024-07-22', TRUE), 
(108, 'Divya',  'Nair',   'divya.n@email.com',  'Kochi',     'Kerala',      '2024-08-30', FALSE); 

-- ========== INSERT: products ========== 
INSERT INTO products VALUES 
(201, 'Wireless Earbuds',     'Electronics', 'BoAt',          1499.00, 250), 
(202, 'Cotton T-Shirt',       'Clothing',    'Levis',         799.00,  500), 
(203, 'Smart Watch',          'Electronics', 'Noise',         2999.00, 150), 
(204, 'Running Shoes',        'Clothing',    'Nike',          4599.00, 120), 
(205, 'Bluetooth Speaker',    'Electronics', 'JBL',           3499.00, 200), 
(206, 'Bedsheet Set',         'Home',        'Spaces',        1299.00, 300), 
(207, 'Laptop Stand',         'Electronics', 'AmazonBasics',  899.00,  180), 
(208, 'Cushion Covers (Set)', 'Home',        'HomeCenter',    599.00,  400); 

-- ========== INSERT: orders ========== 
INSERT INTO orders VALUES 
(1001, 101, '2024-08-01', 'Delivered',  4498.00), 
(1002, 102, '2024-08-03', 'Delivered',  799.00), 
(1003, 103, '2024-08-05', 'Shipped',    7498.00), 
(1004, 101, '2024-08-10', 'Delivered',  3499.00), 
(1005, 104, '2024-08-12', 'Cancelled',  2999.00), 
(1006, 105, '2024-08-15', 'Delivered',  5898.00), 
(1007, 106, '2024-08-18', 'Pending',    1299.00), 
(1008, 103, '2024-08-20', 'Delivered',  899.00), 
(1009, 107, '2024-08-25', 'Shipped',    6098.00), 
(1010, 108, '2024-08-28', 'Delivered',  1598.00); 

-- ========== INSERT: order_items ========== 
INSERT INTO order_items VALUES 
(5001, 1001, 201, 2, 1499.00, 0), 
(5002, 1001, 207, 1, 899.00,  10), 
(5003, 1002, 202, 1, 799.00,  0), 
(5004, 1003, 203, 1, 2999.00, 0), 
(5005, 1003, 204, 1, 4599.00, 5), 
(5006, 1004, 205, 1, 3499.00, 0), 
(5007, 1005, 203, 1, 2999.00, 0), 
(5008, 1006, 201, 1, 1499.00, 10), 
(5009, 1006, 204, 1, 4599.00, 5), 
(5010, 1007, 206, 1, 1299.00, 0), 
(5011, 1008, 207, 1, 899.00,  0), 
(5012, 1009, 205, 1, 3499.00, 0), 
(5013, 1009, 208, 2, 599.00,  15), 
(5014, 1010, 206, 1, 1299.00, 0), 
(5015, 1010, 208, 1, 599.00,  0); 

num_affected_rows,num_inserted_rows
15,15


### Section A — SQL Basics (SELECT, Constraints, Primary Keys) 

Q1. Write a query to display all columns and rows from the customer's table. 

In [0]:
select * from customers;

customer_id,first_name,last_name,email,city,state,join_date,is_premium
101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,true
102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,false
103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,true
104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,false
105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,true
106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,false
107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,true
108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,false


Q2. Retrieve only the first_name, last_name, and city of all customers. 

In [0]:
select first_name, last_name,city from customers;

first_name,last_name,city
Aarav,Sharma,Mumbai
Priya,Patel,Ahmedabad
Rohan,Gupta,Delhi
Sneha,Reddy,Hyderabad
Vikram,Singh,Jaipur
Ananya,Iyer,Chennai
Karan,Mehta,Pune
Divya,Nair,Kochi


Q3. List all unique categories available in the products table. 

In [0]:
select distinct category from products;

category
Electronics
Clothing
Home


 Q4. Identify the Primary Key of each table in the schema. Explain why a Primary Key must be unique and NOT NULL.
## 
 1. Primary Keys in the Schema:
* **`customers` table:** `customer_id`
* **`products` table:** `product_id`
* **`orders` table:** `order_id`
* **`order_items` table:** `item_id`

---

 2. Why a Primary Key must be UNIQUE and NOT NULL:

- It has to be Unique so the database doesn't get confused between rows, and it has to be Not Null so every row actually has an identity!

Q5. What constraints are applied to the email column in the customers table? What would happen if you tried to insert a duplicate email? 
- coconstraints are applied to the email column in the customers table are :-
-   varchar(100)=>string of max length 100
-   unique=>all the strings inserting in the table should be unique no two emails should be same in the table
-   not null=>that means email is must 

Q6. Try inserting a product with unit_price = -50. What happens and which constraint prevents it? Write both the INSERT statement and explain the error. 

In [0]:
--throws an error!!!! 
insert into products values(209,'Head set','electronics','gadget',-50.0,500);

--when i check in the final file the error is showing very big so im clearing the output

- the constraint which prevents insertion of -50 is this :-check(unit_price>0)

### Section B — Filtering & Optimization (WHERE, Indexes) 

Q7. Retrieve all orders with status = 'Delivered'. 

In [0]:
select * from orders where status='Delivered';

order_id,customer_id,order_date,status,total_amount
1001,101,2024-08-01,Delivered,4498.00
1002,102,2024-08-03,Delivered,799.00
1004,101,2024-08-10,Delivered,3499.00
1006,105,2024-08-15,Delivered,5898.00
1008,103,2024-08-20,Delivered,899.00
1010,108,2024-08-28,Delivered,1598.00


Q8. Find all products in the 'Electronics' category with a unit_price greater than ₹2000. 

In [0]:
select * from products where category='Electronics' and unit_price>2000;

product_id,product_name,category,brand,unit_price,stock_qty
203,Smart Watch,Electronics,Noise,2999.00,150
205,Bluetooth Speaker,Electronics,JBL,3499.00,200


Q9. List all customers who joined in the year 2024 and belong to the state 'Maharashtra'. 

In [0]:
select * from customers where year(join_date)=2024 and state='Maharashtra';

customer_id,first_name,last_name,email,city,state,join_date,is_premium
101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,true
107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,true


Q10. Find all orders placed between '2024-08-10' and '2024-08-25' (inclusive) that are NOT cancelled. 

In [0]:
select * from orders where order_date between '2024-08-10' and '2024-08-25' and status != 'Cancelled';

order_id,customer_id,order_date,status,total_amount
1004,101,2024-08-10,Delivered,3499.00
1006,105,2024-08-15,Delivered,5898.00
1007,106,2024-08-18,Pending,1299.00
1008,103,2024-08-20,Delivered,899.00
1009,107,2024-08-25,Shipped,6098.00


Q11. Explain what the index idx_orders_date does. How would it improve the performance of a query that filters orders by order_date? Write a sample query that would benefit from this index. 

### What it does:
 Instead of searching through thousands of rows one by one from top to bottom, the index lets it jump straight to the exact dates you want. This makes your queries run instantly.
`TC:- o(n) to o(logn)` uses b-trees
### Sample Query:
Any query using a `where` clause with a date will run much faster:


In [0]:
select * from orders where order_date = '2024-08-15';

order_id,customer_id,order_date,status,total_amount
1006,105,2024-08-15,Delivered,5898.00


Q12. If you run: SELECT * FROM customers WHERE YEAR(join_date) = 2024; — would the index on join_date be used? Explain why or why not, and rewrite the query to be index-friendly (SARGable). 

No, the index will not be used. When you wrap a column inside a function like YEAR(join_date), MySQL cannot use the index. It forces the database to run that calculation on every single row in the table one by one. To make a query index-friendly (SARGable), you must leave the column completely alone on its side of the operator.

**If you wanna make use of index just use between condition with years start and end dates as shown below **

In [0]:
select * from customers where join_date >= '2024-01-01' and join_date <= '2024-12-31';

customer_id,first_name,last_name,email,city,state,join_date,is_premium
101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,true
102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,false
103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,true
104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,false
105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,true
106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,false
107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,true
108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,false


### Section C — Aggregation (GROUP BY, SUM, COUNT, AVG, MIN, MAX) 

Q13. Count the total number of orders in the orders table. 

In [0]:
select count(*) as total_orders from orders;

total_orders
10


In [0]:
select count(order_id) as total_orders from orders;

total_orders
10


Q14. Find the total revenue (SUM of total_amount) from all 'Delivered' orders. 

In [0]:
select sum(total_amount) as total_delivered_revenue from orders
where status = 'Delivered';

total_delivered_revenue
17191.00


Q15. Calculate the average unit_price of products in each category. 

In [0]:
select category, round(avg(unit_price)) as average_price
from products
group by category;

category,average_price
Electronics,2224
Clothing,2699
Home,949


Q16. For each order status, find the count of orders and the total revenue. Sort the result by total revenue in descending order. 

In [0]:
select status,count(*) as order_count,sum(total_amount) as total_revenue
from orders
group by status
order by total_revenue desc;

status,order_count,total_revenue
Delivered,6,17191.00
Shipped,2,13596.00
Cancelled,1,2999.00
Pending,1,1299.00


Q17. Find the most expensive (MAX) and cheapest (MIN) product in each category. 

In [0]:
select category,max(unit_price) as most_expensive,min(unit_price) as cheapest
from products
group by category;

category,most_expensive,cheapest
Electronics,3499.00,899.00
Clothing,4599.00,799.00
Home,1299.00,599.00


Q18. List all product categories where the average unit_price is greater than ₹2000. (Hint: Use HAVING clause)

In [0]:
select category, avg(unit_price) as average_price
from products
group by category
having avg(unit_price) > 2000;

category,average_price
Electronics,2224.000000
Clothing,2699.000000


## Section D — Joins & Relationships 

Q19. Write an INNER JOIN query to display each order along with the customer's first_name and last_name. Show: order_id, order_date, first_name, last_name, total_amount. 

In [0]:
select o.order_id, o.order_date, c.first_name, c.last_name, 
o.total_amount
from customers c
inner join orders o on c.customer_id = o.customer_id;

order_id,order_date,first_name,last_name,total_amount
1004,2024-08-10,Aarav,Sharma,3499.00
1002,2024-08-03,Priya,Patel,799.00
1008,2024-08-20,Rohan,Gupta,899.00
1005,2024-08-12,Sneha,Reddy,2999.00
1006,2024-08-15,Vikram,Singh,5898.00
1007,2024-08-18,Ananya,Iyer,1299.00
1009,2024-08-25,Karan,Mehta,6098.00
1010,2024-08-28,Divya,Nair,1598.00
1001,2024-08-01,Aarav,Sharma,4498.00
1003,2024-08-05,Rohan,Gupta,7498.00


Q20. Using a LEFT JOIN, list ALL customers and their orders (if any). Customers with no orders should still appear with NULL values for order columns. 

In [0]:
select c.customer_id,c.first_name, c.last_name, o.order_id, 
o.order_date, o.total_amount
from customers c
left join orders o on c.customer_id = o.customer_id;

customer_id,first_name,last_name,order_id,order_date,total_amount
101,Aarav,Sharma,1004,2024-08-10,3499.00
102,Priya,Patel,1002,2024-08-03,799.00
103,Rohan,Gupta,1008,2024-08-20,899.00
104,Sneha,Reddy,1005,2024-08-12,2999.00
105,Vikram,Singh,1006,2024-08-15,5898.00
106,Ananya,Iyer,1007,2024-08-18,1299.00
107,Karan,Mehta,1009,2024-08-25,6098.00
108,Divya,Nair,1010,2024-08-28,1598.00
101,Aarav,Sharma,1001,2024-08-01,4498.00
103,Rohan,Gupta,1003,2024-08-05,7498.00


Q21. Write a query using JOINs across three tables (orders → order_items → products) to show: order_id, product_name, quantity, unit_price, and discount_pct for each order item. 

In [0]:
select oi.order_id, p.product_name, oi.quantity, oi.unit_price, 
oi.discount_pct
from orders o
join order_items oi on o.order_id = oi.order_id
join products p on oi.product_id = p.product_id;

order_id,product_name,quantity,unit_price,discount_pct
1001,Wireless Earbuds,2,1499.00,0.00
1001,Laptop Stand,1,899.00,10.00
1002,Cotton T-Shirt,1,799.00,0.00
1003,Smart Watch,1,2999.00,0.00
1003,Running Shoes,1,4599.00,5.00
1004,Bluetooth Speaker,1,3499.00,0.00
1005,Smart Watch,1,2999.00,0.00
1006,Wireless Earbuds,1,1499.00,10.00
1006,Running Shoes,1,4599.00,5.00
1007,Bedsheet Set,1,1299.00,0.00


Q22. Explain the difference between LEFT JOIN and RIGHT JOIN with an example from this schema. When would you use a FULL OUTER JOIN? 

LEFT JOIN vs RIGHT JOIN

LEFT JOIN → Keep all rows from the left table. If there is no match in the right table, the right-side columns become NULL.
RIGHT JOIN → Keep all rows from the right table. If there is no match in the left table, the left-side columns become NULL.
When to use FULL OUTER JOIN?

Use FULL OUTER JOIN when you want to see everything from both tables, whether the rows match or not.

It shows:

Rows that exist in both tables.
Rows that exist only in the first table.
Rows that exist only in the second table.

In [0]:
select c.customer_id, c.first_name, o.order_id, o.status
from customers c
left join orders o on c.customer_id = o.customer_id;.

customer_id,first_name,order_id,status
101,Aarav,1004,Delivered
102,Priya,1002,Delivered
103,Rohan,1008,Delivered
104,Sneha,1005,Cancelled
105,Vikram,1006,Delivered
106,Ananya,1007,Pending
107,Karan,1009,Shipped
108,Divya,1010,Delivered
101,Aarav,1001,Delivered
103,Rohan,1003,Shipped


In [0]:
select c.customer_id, o.order_id
from customers c
left join orders o on c.customer_id = o.customer_id
union
select c.customer_id, o.order_id
from customers c
right join orders o on c.customer_id = o.customer_id;

customer_id,order_id
101,1004
103,1008
101,1001
102,1002
104,1005
108,1010
107,1009
105,1006
103,1003
106,1007


Q23. Identify all Foreign Key relationships in the schema. Explain what would happen if you tried to insert an order with customer_id = 999 (which doesn't exist in customers).

There are three Foreign Key (FK) relationships:-      
`orders.customer_id` → `customers.customer_id`

`order_items.order_id` → `orders.order_id`

`order_items.product_id` →` products.product_id`

2. What happens if you insert an order with customer_id = 999?
- Insertion fails with a Foreign Key error because the parent record doesn't exist.

## Section E — Advanced Concepts (CASE, ACID, Transactions) 

Q24. Write a query using CASE to classify products into price tiers: 
  • 'Budget'    → unit_price < 1000 
  • 'Mid-Range' → unit_price BETWEEN 1000 AND 3000 
  • 'Premium'   → unit_price > 3000 
Display: product_name, unit_price, price_tier. 

In [0]:
select product_name, unit_price,
    case 
        when unit_price < 1000 then 'Budget'
        when unit_price between 1000 and 3000 then 'Mid-Range'
        else 'Premium'
    end as price_tier
from products;

product_name,unit_price,price_tier
Wireless Earbuds,1499.00,Mid-Range
Cotton T-Shirt,799.00,Budget
Smart Watch,2999.00,Mid-Range
Running Shoes,4599.00,Premium
Bluetooth Speaker,3499.00,Premium
Bedsheet Set,1299.00,Mid-Range
Laptop Stand,899.00,Budget
Cushion Covers (Set),599.00,Budget


Q25. Using a CASE statement inside an aggregate function, count how many orders are 'Delivered' vs 'Not Delivered' (all other statuses). Display the result in a s

In [0]:
select sum(case when status = 'delivered' then 1 else 0 end) as delivered_count,
sum(case when status != 'delivered' then 1 else 0 end) as not_delivered_count
from orders;

delivered_count,not_delivered_count
0,10


Q26. Explain each letter of ACID: 
  • A – Atomicity 
  • C – Consistency 
  • I – Isolation 
  • D – Durability 
Give a real-world example (e.g., bank transfer) showing why each property is important. 

ACID Properties (Bank Transfer Example)

 **A – Atomicity**
- Either all steps happen or none happen.
- Example: Money is deducted from A and added to B together. If one step fails, both are cancelled.
**C – Consistency**
- Data remains correct before and after the transaction.
- Example: Total money in both accounts stays the same after the transfer.
**I – Isolation**
- Multiple transactions do not interfere with each other.
- Example: Two people transferring money at the same time won't affect each other's transactions.
**D – Durability**
- Once a transaction is committed, it is permanently saved.
- Example: After a successful transfer, the updated balance remains even if the system crashes.

Q27. Write a SQL transaction that does the following atomically: 
  1. Insert a new order (order_id=1011, customer_id=102, today's date, 'Pending', 1598.00) 
  2. Insert two order items for that order 
  3. Update the stock_qty of the purchased products 
  4. If any step fails, ROLLBACK the entire transaction. Otherwise, COMMIT. 
Write the complete BEGIN...COMMIT/ROLLBACK block. 

In [0]:
begin transaction;

-- 1.
insert into orders (order_id, customer_id, order_date, status, total_amount)
values (1011, 102, current_date(), 'Pending', 1598.00);

-- 2.
insert into order_items (item_id, order_id, product_id, quantity, unit_price, discount_pct)
values 
(5016, 1011, 206, 1, 1299.00, 0),
(5017, 1011, 203, 1, 299.00, 0);

-- 3.
update products 
set stock_qty = stock_qty - 1 
where product_id in (203, 206);
commit;

num_affected_rows,num_inserted_rows
2,2
